In [ ]:
# The mechanistic justification for this step is: [Environment Setup]
# Install required libraries and set up Kaggle pathing invariant and PyTorch backend.

!pip install datasketch transformers pyarrow pandas scikit-learn

import os
import json
import re
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModel
from datasketch import MinHash, MinHashLSH

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Kaggle Pathing Invariant
BASE_PATH = '/kaggle/input/datasets/murtadhayaseen/arabic-fake-news-dataset-afnd/AFND'
OUTPUT_DIR = '/kaggle/working'
os.makedirs(OUTPUT_DIR, exist_ok=True)


In [ ]:
# The mechanistic justification for this step is: [Data Loading]
# Load the raw AFND dataset from the Kaggle input directory into a pandas DataFrame.

print("Loading AFND Dataset...")
sources_file = os.path.join(BASE_PATH, "sources.json")
    
with open(sources_file, "r", encoding="utf-8") as f:
    sources_mapping = json.load(f)
    
records = []
missing = 0
for source_id, label in sources_mapping.items():
    articles_file = os.path.join(BASE_PATH, "Dataset", str(source_id), "scraped_articles.json")
    if not os.path.exists(articles_file):
        missing += 1
        continue
        
    with open(articles_file, "r", encoding="utf-8") as f:
        data = json.load(f)
        articles = data["articles"] if isinstance(data, dict) and "articles" in data else data
            
    for article in articles:
        records.append({
            "source": str(source_id),
            "label": label,
            "title": str(article.get("title", "")),
            "text": str(article.get("text", ""))
        })
        
df = pd.DataFrame(records)
print(f"Total raw articles loaded: {len(df)}")
if missing:
    print(f"Missing source directories: {missing}")


In [ ]:
# The mechanistic justification for this step is: [Data Cleaning]
# Filter out junk articles and run LSH to remove near-duplicates to prevent data leakage.

print("Filtering junk articles...")
# Drop empty text & short articles
df = df[df['text'].str.strip().astype(bool)]
df = df[df['text'].apply(lambda x: len(str(x).split()) >= 10)]

def html_density(text):
    text = str(text)
    if not text: return 0
    html_len = sum(len(m.group()) for m in re.finditer(r'<[^>]+>', text))
    return html_len / len(text)
    
df = df[df['text'].apply(html_density) <= 0.3]

# Exact duplicates
df['hash_key'] = df['title'].str.strip() + " | " + df['text'].str.strip()
df = df.drop_duplicates(subset=['hash_key']).drop(columns=['hash_key']).reset_index(drop=True)
print(f"Articles after basic cleaning: {len(df)}")

print("Running LSH near-duplicate detection... (This may take ~10-20 minutes on Kaggle CPU)")
def clean_text_lsh(text):
    return ' '.join(re.sub(r'<[^>]+>', ' ', str(text)).split())

lsh = MinHashLSH(threshold=0.8, num_perm=128)
minhashes = {}

for idx, row in df.iterrows():
    text = clean_text_lsh(str(row['title']) + " " + str(row['text']))
    m = MinHash(num_perm=128)
    words = text.split()
    for i in range(len(words) - 4):
        m.update(" ".join(words[i:i+5]).encode('utf8'))
    lsh.insert(str(idx), m)
    minhashes[str(idx)] = m

dropped_indices = set()
visited = set()
for idx in df.index:
    str_idx = str(idx)
    if str_idx in visited: continue
    result = lsh.query(minhashes[str_idx])
    if len(result) > 1:
        for r in result:
            visited.add(r)
            if r != str_idx: dropped_indices.add(int(r))
    else:
        visited.add(str_idx)

df = df.drop(index=list(dropped_indices)).reset_index(drop=True)
print(f"Articles after LSH near-duplicate dropping: {len(df)}")


In [ ]:
# The mechanistic justification for this step is: [Source-Disjoint Split Generation]
# Create training and evaluation splits that have zero source overlap to test for Source-Induced Credibility.

print("Generating Source-Disjoint Splits...")
np.random.seed(42)
source_stats = df.groupby(['source', 'label']).size().reset_index(name='count')
total_articles = source_stats['count'].sum()

best_score = float('inf')
best_allocation = {}

for _ in range(200):
    allocation = {}
    counts = {'train': 0, 'val': 0, 'test': 0}
    shuffled = source_stats.sample(frac=1.0)
    
    for _, row in shuffled.iterrows():
        src = row['source']
        c = row['count']
        
        train_ratio = counts['train'] / max(1, sum(counts.values()) + c)
        val_ratio = counts['val'] / max(1, sum(counts.values()) + c)
        test_ratio = counts['test'] / max(1, sum(counts.values()) + c)
        
        diffs = {'train': 0.64 - train_ratio, 'val': 0.16 - val_ratio, 'test': 0.20 - test_ratio}
        assigned = max(diffs, key=diffs.get)
        allocation[src] = assigned
        counts[assigned] += c
        
    score = abs(counts['train']/total_articles - 0.64) + abs(counts['val']/total_articles - 0.16) + abs(counts['test']/total_articles - 0.20)
    if score < best_score:
        best_score = score
        best_allocation = allocation

df['split'] = df['source'].map(best_allocation)

train_df = df[df['split'] == 'train'].drop(columns=['split'])
val_df = df[df['split'] == 'val'].drop(columns=['split'])
test_df = df[df['split'] == 'test'].drop(columns=['split'])

label_map = {l: i for i, l in enumerate(train_df['label'].unique())}
reverse_label_map = {i: l for l, i in label_map.items()}

for d in [train_df, val_df, test_df]:
    d['label_id'] = d['label'].map(label_map)

print(f"Disjoint Split - Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

# Save splits for future use
train_df.to_parquet(f'{OUTPUT_DIR}/train_src.parquet')
val_df.to_parquet(f'{OUTPUT_DIR}/val_src.parquet')
test_df.to_parquet(f'{OUTPUT_DIR}/test_src.parquet')


In [ ]:
# The mechanistic justification for this step is: [Model & Dataset Initialization]
# Prepare AraBERTv0.2 PyTorch architecture and DataLoaders for memory-efficient training.

class TransformerDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts.tolist()
        self.labels = labels.tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len
        
    def __len__(self): return len(self.texts)
        
    def __getitem__(self, idx):
        encoding = self.tokenizer(str(self.texts[idx]), add_special_tokens=True, max_length=self.max_len, padding='max_length', truncation=True, return_attention_mask=True, return_tensors='pt')
        return {'input_ids': encoding['input_ids'].flatten(), 'attention_mask': encoding['attention_mask'].flatten(), 'labels': torch.tensor(self.labels[idx], dtype=torch.long)}

class TransformerClassifier(nn.Module):
    def __init__(self, model_name, num_classes):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(self.bert.config.hidden_size, num_classes)
        
    def forward(self, input_ids, attention_mask):
        return self.classifier(self.dropout(self.bert(input_ids=input_ids, attention_mask=attention_mask).pooler_output))

model_name = "aubmindlab/bert-base-arabertv02"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Sample train set to speed up Kaggle training (optional, but LSH already reduced the set)
train_sample = train_df.sample(n=min(len(train_df), 100000), random_state=42)

train_trans_ds = TransformerDataset(train_sample['title'] + " " + train_sample['text'], train_sample['label_id'], tokenizer)
val_trans_ds = TransformerDataset(val_df['title'] + " " + val_df['text'], val_df['label_id'], tokenizer)
test_trans_ds = TransformerDataset(test_df['title'] + " " + test_df['text'], test_df['label_id'], tokenizer)

train_loader = DataLoader(train_trans_ds, batch_size=16, shuffle=True) 
val_loader = DataLoader(val_trans_ds, batch_size=16, shuffle=False)
test_loader = DataLoader(test_trans_ds, batch_size=16, shuffle=False)

arabert = TransformerClassifier(model_name=model_name, num_classes=len(label_map)).to(device)


In [ ]:
# The mechanistic justification for this step is: [Extended Training Loop]
# Train AraBERT on the disjoint split for 10 epochs, saving the best generalization weights.

optimizer = torch.optim.AdamW(arabert.parameters(), lr=2e-5)
loss_fn = nn.CrossEntropyLoss()

epochs = 10
best_val_loss = float('inf')

print("Starting AraBERT Training...")
for epoch in range(epochs):
    print(f"\nEpoch {epoch+1}/{epochs}")
    arabert.train()
    total_train_loss = 0
    for batch_idx, batch in enumerate(train_loader):
        input_ids, mask, labels = batch['input_ids'].to(device), batch['attention_mask'].to(device), batch['labels'].to(device)
        optimizer.zero_grad()
        loss = loss_fn(arabert(input_ids, mask), labels)
        loss.backward()
        optimizer.step()
        total_train_loss += loss.item()
        
        if batch_idx > 0 and batch_idx % 1000 == 0:
            print(f"  Batch {batch_idx}/{len(train_loader)} | Loss: {loss.item():.4f}")
            
    print(f"Avg Train Loss: {total_train_loss / len(train_loader):.4f}")
    
    arabert.eval()
    total_val_loss = 0
    correct, total = 0, 0
    with torch.no_grad():
        for batch in val_loader:
            input_ids, mask, labels = batch['input_ids'].to(device), batch['attention_mask'].to(device), batch['labels'].to(device)
            outputs = arabert(input_ids, mask)
            loss = loss_fn(outputs, labels)
            total_val_loss += loss.item()
            correct += (torch.argmax(outputs, dim=1) == labels).sum().item()
            total += labels.size(0)
            
    avg_val_loss = total_val_loss / len(val_loader)
    print(f"Avg Val Loss: {avg_val_loss:.4f} | Val Accuracy: {correct/total:.4f}")
    
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(arabert.state_dict(), f"{OUTPUT_DIR}/arabert_best_disjoint.pt")
        print(f"Saved new best model to {OUTPUT_DIR}/arabert_best_disjoint.pt")


In [ ]:
# The mechanistic justification for this step is: [Error Taxonomy Extraction]
# Evaluate on the unseen test split, extract errors, and categorize them to document Source-Induced Credibility collapse.

print("Evaluating on Test Set...")
arabert.load_state_dict(torch.load(f"{OUTPUT_DIR}/arabert_best_disjoint.pt"))
arabert.eval()

all_preds = []
correct, total = 0, 0
with torch.no_grad():
    for batch in test_loader:
        input_ids, mask, labels = batch['input_ids'].to(device), batch['attention_mask'].to(device), batch['labels'].to(device)
        outputs = arabert(input_ids, mask)
        preds = torch.argmax(outputs, dim=1).cpu().numpy()
        all_preds.extend(preds)
        correct += (preds == labels.cpu().numpy()).sum()
        total += labels.size(0)

print(f"Test Accuracy: {correct/total:.4f}")

test_df['pred_id'] = all_preds
test_df['pred'] = test_df['pred_id'].map(reverse_label_map)

errors_df = test_df[test_df['label'] != test_df['pred']]
print(f"Total Errors Found: {len(errors_df)} out of {len(test_df)}")

def categorize_error(row):
    full_text = str(row.get('title', '')) + " " + str(row.get('text', ''))
    if any(a in full_text for a in ['النهار أونلاين', 'وكالة الأنباء', 'رويترز', 'CNN', 'الجزيرة']): return "Source-Specific Artifacts"
    if any(w in full_text for w in ['نفى', 'رئيس', 'وزير', 'الحكومة', 'رسمي']): return "Government & Official Statements"
    if any(w in full_text for w in ['كورونا', 'صحة', 'مستشفى', 'فيروس']): return "Health & COVID-19"
    if any(w in full_text for w in ['دينار', 'دولار', 'أسعار', 'اقتصاد']): return "Economics & Pricing"
    if any(w in full_text for w in ['ملعب', 'مباراة', 'فريق', 'بطولة']): return "Sports News"
    if any(w in full_text for w in ['شرطة', 'أمن', 'اعتقال', 'جريمة']): return "Crime & Accidents"
    return "General/Other Ambiguous Context"

errors_df['error_category'] = errors_df.apply(categorize_error, axis=1)

summary = errors_df.groupby(['error_category']).size().reset_index(name='count').sort_values(by='count', ascending=False)
with open(f"{OUTPUT_DIR}/taxonomy_report.md", "w", encoding="utf-8") as f:
    f.write("# Error Taxonomy Summary (Disjoint Split)\n\n")
    for _, row in summary.iterrows():
        f.write(f"- **{row['error_category']}**: {row['count']} errors\n")

print(f"Taxonomy saved to {OUTPUT_DIR}/taxonomy_report.md")
